### 01. Analysis Setup

**Business objective:**  
This analysis uses flight-level operational data to understand where delays, cancellations, and operational disruption occur across time, airlines, airports, delay causes, and seasons.

We begin by loading the dataset and creating a SQL-based analysis environment so the large flight dataset can be explored efficiently and consistently.


In [13]:
import numpy as np
import pandas as pd
import duckdb

file_path = r"C:\Users\Sahil\OneDrive\Desktop\jupyter lab\Project 5\flights_sample_3m.csv"

con = duckdb.connect()

## 02. Establish the Dataset & Coverage

Before analyzing performance, we validate the dataset's scope and coverage. We check the number of records and the available date range so that the conclusions are grounded in the actual period represented in the data.

This also establishes whether the dataset is suitable for comparing operational performance across years.


In [14]:
con.execute(f"""
    CREATE VIEW flights AS
    SELECT *
    FROM read_csv_auto('{file_path}')
""")

con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        MIN(FL_DATE) AS earliest_date,
        MAX(FL_DATE) AS latest_date,
        COUNT(DISTINCT YEAR(FL_DATE)) AS years_in_data
    FROM flights
""").df()

,total_rows,earliest_date,latest_date,years_in_data
0,3000000,2019-01-01,2023-08-31,5


## 03. Understand the Time Coverage

**Question: How is flight activity distributed across the available years?**

We first quantify flight volume by year. This gives us the baseline needed to interpret later changes in cancellations, delays, and operational performance without confusing a change in flight volume with a change in performance.


In [15]:
con.sql("""
    SELECT
        YEAR(FL_DATE) AS year,
        COUNT(*) AS flight_count
    FROM flights
    GROUP BY year
    ORDER BY year
""").df()

,year,flight_count
0,2019,757673
1,2020,479350
2,2021,611633
3,2022,687860
4,2023,463484


## 04. Understand the Available Variables

We inspect the table structure and data types to identify the fields required for the analysis, including flight dates, airline, origin and destination airports, delays, cancellations, diversions, and specific delay causes.

This step ensures the following metrics are built from the correct operational fields.


In [16]:
con.sql("DESCRIBE flights").df()

,column_name,column_type,null,key,default,extra
0,FL_DATE,DATE,YES,None,None,None
1,AIRLINE,VARCHAR,YES,None,None,None
2,AIRLINE_DOT,VARCHAR,YES,None,None,None
3,AIRLINE_CODE,VARCHAR,YES,None,None,None
4,DOT_CODE,BIGINT,YES,None,None,None
5,FL_NUMBER,BIGINT,YES,None,None,None
6,ORIGIN,VARCHAR,YES,None,None,None
7,ORIGIN_CITY,VARCHAR,YES,None,None,None
8,DEST,VARCHAR,YES,None,None,None
9,DEST_CITY,VARCHAR,YES,None,None,None


## 05. Inspect Flight-Level Records

We review individual flight records before creating analytical metrics. The goal is to understand how operational outcomes and delay information are represented at the flight level and confirm which fields can be used to classify completed, cancelled, and delayed flights.


In [17]:
con.sql("""
    SELECT
        FL_DATE,
        AIRLINE,
        ORIGIN,
        DEST,
        DEP_DELAY,
        ARR_DELAY,
        CANCELLED,
        DIVERTED,
        DELAY_DUE_CARRIER,
        DELAY_DUE_WEATHER,
        DELAY_DUE_NAS,
        DELAY_DUE_LATE_AIRCRAFT
    FROM flights
    WHERE YEAR(FL_DATE) IN (2019, 2022)
    LIMIT 10
""").df()

,FL_DATE,AIRLINE,ORIGIN,DEST,DEP_DELAY,ARR_DELAY,CANCELLED,DIVERTED,DELAY_DUE_CARRIER,DELAY_DUE_WEATHER,DELAY_DUE_NAS,DELAY_DUE_LATE_AIRCRAFT
0,2019-01-09,United Air Lines Inc.,FLL,EWR,-4.0,-14.0,0.0,0.0,NaN,NaN,NaN,NaN
1,2022-11-19,Delta Air Lines Inc.,MSP,SEA,-6.0,-5.0,0.0,0.0,NaN,NaN,NaN,NaN
2,2022-07-22,United Air Lines Inc.,DEN,MSP,6.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
3,2019-07-31,Southwest Airlines Co.,DAL,OKC,147.0,141.0,0.0,0.0,141.0,0.0,0.0,0.0
4,2019-07-08,Republic Airline,HSV,DCA,-6.0,23.0,0.0,0.0,0.0,0.0,23.0,0.0
5,2019-11-20,Delta Air Lines Inc.,BDL,ATL,-5.0,-32.0,0.0,0.0,NaN,NaN,NaN,NaN
6,2022-05-01,Southwest Airlines Co.,BWI,BDL,3.0,6.0,0.0,0.0,NaN,NaN,NaN,NaN
7,2019-03-24,Spirit Air Lines,DEN,IAH,-4.0,-13.0,0.0,0.0,NaN,NaN,NaN,NaN
8,2019-03-17,United Air Lines Inc.,SRQ,ORD,-4.0,35.0,0.0,0.0,0.0,0.0,35.0,0.0
9,2022-05-05,JetBlue Airways,JFK,CHS,-3.0,-26.0,0.0,0.0,NaN,NaN,NaN,NaN


## 06. Create an Analysis-Ready Flight View

To make the analysis consistent, we create a derived flight-level view for the selected analysis period. This adds reusable fields such as flight year, arrival status, completion flags, and delay indicators.

Creating these fields once allows the following analyses to use the same definitions and makes the results easier to reproduce.


In [21]:
con.execute("""
    CREATE OR REPLACE VIEW flights_2019_2022 AS
    SELECT
        *,
        YEAR(FL_DATE) AS flight_year,

        CASE
            WHEN CANCELLED = 1 THEN 'Cancelled'
            WHEN DIVERTED = 1 THEN 'Diverted'
            WHEN ARR_DELAY IS NULL THEN 'Missing arrival result'
            WHEN ARR_DELAY < 15 THEN 'On time'
            ELSE 'Delayed'
        END AS arrival_status,

        CASE
            WHEN CANCELLED = 0
             AND DIVERTED = 0
             AND ARR_DELAY IS NOT NULL
            THEN 1
            ELSE 0
        END AS completed_flight_flag,

        CASE
            WHEN CANCELLED = 0
             AND DIVERTED = 0
             AND ARR_DELAY >= 15
            THEN 1
            ELSE 0
        END AS delayed_arrival_flag

    FROM flights
    WHERE YEAR(FL_DATE) IN (2019, 2022)
""")

## 07. Question: How Does Flight Performance Change Over Time?

We begin the performance analysis by comparing the number of flights across arrival-status categories for each year. This shows whether operational disruption is becoming more or less common over time and provides context for the deeper delay and cancellation analysis that follows.


In [22]:
con.sql("""
    SELECT
        flight_year,
        arrival_status,
        COUNT(*) AS flights
    FROM flights_2019_2022
    GROUP BY flight_year, arrival_status
    ORDER BY flight_year, arrival_status
""").df()

,flight_year,arrival_status,flights
0,2019,Cancelled,13594
1,2019,Delayed,141253
2,2019,Diverted,1983
3,2019,On time,600843
4,2022,Cancelled,18448
5,2022,Delayed,140468
6,2022,Diverted,1650
7,2022,Missing arrival result,1
8,2022,On time,527293


## 08. Measure the Overall Operational Baseline

We quantify scheduled flights, cancellations, completed flights, and delay-related metrics by year. The objective is to establish the overall operational baseline and identify years where disruption was particularly high or low.

These figures provide the benchmark against which airline and airport performance can be compared.


In [23]:
con.sql("""
    SELECT
        flight_year,
        COUNT(*) AS total_scheduled_flights,

        SUM(CASE WHEN CANCELLED = 1 THEN 1 ELSE 0 END) AS cancelled_flights,
        ROUND(
            100.0 * SUM(CASE WHEN CANCELLED = 1 THEN 1 ELSE 0 END)
            / COUNT(*),
            2
        ) AS cancellation_rate_pct,

        SUM(completed_flight_flag) AS completed_flights,

        ROUND(
            100.0 * SUM(
                CASE
                    WHEN completed_flight_flag = 1
                     AND arrival_status = 'On time'
                    THEN 1
                    ELSE 0
                END
            ) / NULLIF(SUM(completed_flight_flag), 0),
            2
        ) AS on_time_arrival_rate_pct,

        ROUND(
            100.0 * SUM(delayed_arrival_flag)
            / NULLIF(SUM(completed_flight_flag), 0),
            2
        ) AS delayed_arrival_rate_pct,

        ROUND(
            AVG(
                CASE
                    WHEN completed_flight_flag = 1
                    THEN ARR_DELAY
                END
            ),
            2
        ) AS average_arrival_delay_minutes

    FROM flights_2019_2022
    GROUP BY flight_year
    ORDER BY flight_year
""").df()

,flight_year,total_scheduled_flights,cancelled_flights,cancellation_rate_pct,completed_flights,on_time_arrival_rate_pct,delayed_arrival_rate_pct,average_arrival_delay_minutes
0,2019,757673,13594.0,1.79,742096.0,80.97,19.03,5.31
1,2022,687860,18448.0,2.68,667761.0,78.96,21.04,6.91


## 09. Question: Which Airlines Have the Strongest Operational Performance?

We compare airlines on completed-flight volume and arrival-delay performance by year. Looking at both scale and reliability prevents us from judging an airline solely on the number of flights it operates.

The goal is to identify meaningful differences in operational performance between carriers.


In [24]:
con.sql("""
    WITH airline_yearly AS (
        SELECT
            flight_year,
            AIRLINE,
            SUM(completed_flight_flag) AS completed_flights,

            ROUND(
                100.0 * SUM(
                    CASE
                        WHEN completed_flight_flag = 1
                         AND arrival_status = 'On time'
                        THEN 1
                        ELSE 0
                    END
                ) / NULLIF(SUM(completed_flight_flag), 0),
                2
            ) AS on_time_rate_pct,

            ROUND(
                AVG(
                    CASE
                        WHEN completed_flight_flag = 1
                        THEN ARR_DELAY
                    END
                ),
                2
            ) AS average_arrival_delay_minutes

        FROM flights_2019_2022
        GROUP BY flight_year, AIRLINE
    )

    SELECT
        a2019.AIRLINE,
        a2019.completed_flights AS completed_flights_2019,
        a2022.completed_flights AS completed_flights_2022,
        a2019.on_time_rate_pct AS on_time_rate_2019,
        a2022.on_time_rate_pct AS on_time_rate_2022,
        ROUND(
            a2022.on_time_rate_pct - a2019.on_time_rate_pct,
            2
        ) AS on_time_rate_change_pp,
        a2019.average_arrival_delay_minutes AS avg_delay_2019,
        a2022.average_arrival_delay_minutes AS avg_delay_2022

    FROM airline_yearly a2019
    INNER JOIN airline_yearly a2022
        ON a2019.AIRLINE = a2022.AIRLINE
    WHERE a2019.flight_year = 2019
      AND a2022.flight_year = 2022
      AND a2019.completed_flights >= 10000
      AND a2022.completed_flights >= 10000
    ORDER BY on_time_rate_change_pp ASC
""").df()

,AIRLINE,completed_flights_2019,completed_flights_2022,on_time_rate_2019,on_time_rate_2022,on_time_rate_change_pp,avg_delay_2019,avg_delay_2022
0,Allegiant Air,10729.0,11562.0,79.59,66.96,-12.63,7.36,19.53
1,JetBlue Airways,29910.0,26649.0,74.75,66.87,-7.88,11.14,18.74
2,Frontier Airlines Inc.,13690.0,15361.0,74.61,67.61,-7.00,9.35,17.38
3,Southwest Airlines Co.,135727.0,129287.0,82.44,75.74,-6.70,2.67,7.19
4,Spirit Air Lines,20493.0,22855.0,80.96,75.53,-5.43,5.09,10.06
5,American Airlines Inc.,94391.0,85962.0,79.14,77.61,-1.53,7.01,9.74
6,Alaska Airlines Inc.,26698.0,22694.0,81.28,79.75,-1.53,1.28,3.61
7,Delta Air Lines Inc.,101044.0,89675.0,85.28,84.09,-1.19,1.40,2.21
8,Republic Airline,32763.0,30684.0,80.75,80.25,-0.50,4.70,4.55
9,PSA Airlines Inc.,28663.0,21197.0,80.40,80.55,0.15,6.44,6.06


## 10. Question: Which Airports Face the Highest Delay Exposure?

We evaluate origin airports using completed-flight volume and delayed-arrival measures. This helps identify airports where operational disruption may be concentrated and where capacity, scheduling, or network conditions could have a larger effect on overall performance.


In [25]:
con.sql("""
    SELECT
        ORIGIN AS origin_airport,
        ORIGIN_CITY AS origin_city,

        SUM(completed_flight_flag) AS completed_flights,

        SUM(delayed_arrival_flag) AS delayed_flights,

        ROUND(
            100.0 * SUM(delayed_arrival_flag)
            / NULLIF(SUM(completed_flight_flag), 0),
            2
        ) AS delayed_arrival_rate_pct,

        ROUND(
            AVG(
                CASE
                    WHEN completed_flight_flag = 1
                    THEN ARR_DELAY
                END
            ),
            2
        ) AS average_arrival_delay_minutes

    FROM flights_2019_2022
    WHERE flight_year = 2022
      AND ORIGIN IS NOT NULL
    GROUP BY ORIGIN, ORIGIN_CITY
    HAVING SUM(completed_flight_flag) >= 5000
    ORDER BY delayed_flights DESC
    LIMIT 15
""").df()

,origin_airport,origin_city,completed_flights,delayed_flights,delayed_arrival_rate_pct,average_arrival_delay_minutes
0,DEN,"Denver, CO",26873.0,6954.0,25.88,10.51
1,DFW,"Dallas/Fort Worth, TX",27160.0,6373.0,23.46,10.56
2,ATL,"Atlanta, GA",31940.0,5782.0,18.10,4.49
3,ORD,"Chicago, IL",25712.0,5439.0,21.15,6.22
4,LAS,"Las Vegas, NV",17291.0,4600.0,26.60,11.28
5,MCO,"Orlando, FL",14534.0,4195.0,28.86,13.75
6,CLT,"Charlotte, NC",18800.0,3967.0,21.10,6.24
7,LGA,"New York, NY",16443.0,3852.0,23.43,7.43
8,JFK,"New York, NY",13447.0,3481.0,25.89,9.39
9,PHX,"Phoenix, AZ",16545.0,3452.0,20.86,6.50


## 11. Prioritize Airport-Level Delay Risk

The airport-level results are further focused on delay rates rather than volume alone. This distinction is important because a large airport will naturally generate more delayed flights; the business question is whether an airport experiences an unusually high *share* of delayed operations.


In [26]:
con.sql("""
    SELECT
        ORIGIN AS origin_airport,
        ORIGIN_CITY AS origin_city,

        SUM(completed_flight_flag) AS completed_flights,
        SUM(delayed_arrival_flag) AS delayed_flights,

        ROUND(
            100.0 * SUM(delayed_arrival_flag)
            / NULLIF(SUM(completed_flight_flag), 0),
            2
        ) AS delayed_arrival_rate_pct,

        ROUND(
            AVG(
                CASE
                    WHEN completed_flight_flag = 1
                    THEN ARR_DELAY
                END
            ),
            2
        ) AS average_arrival_delay_minutes

    FROM flights_2019_2022
    WHERE flight_year = 2022
      AND ORIGIN IS NOT NULL
    GROUP BY ORIGIN, ORIGIN_CITY
    HAVING SUM(completed_flight_flag) >= 5000
    ORDER BY delayed_arrival_rate_pct DESC, delayed_flights DESC
    LIMIT 15
""").df()

,origin_airport,origin_city,completed_flights,delayed_flights,delayed_arrival_rate_pct,average_arrival_delay_minutes
0,MDW,"Chicago, IL",7698.0,2337.0,30.36,11.75
1,MCO,"Orlando, FL",14534.0,4195.0,28.86,13.75
2,BWI,"Baltimore, MD",8551.0,2454.0,28.70,11.25
3,FLL,"Fort Lauderdale, FL",8268.0,2205.0,26.67,12.12
4,LAS,"Las Vegas, NV",17291.0,4600.0,26.60,11.28
5,EWR,"Newark, NJ",12644.0,3317.0,26.23,10.82
6,JFK,"New York, NY",13447.0,3481.0,25.89,9.39
7,DEN,"Denver, CO",26873.0,6954.0,25.88,10.51
8,MIA,"Miami, FL",10605.0,2724.0,25.69,11.22
9,DAL,"Dallas, TX",6620.0,1649.0,24.91,7.02


## 12. Question: What Are the Main Causes of Delay?

Not all delay minutes have the same operational root cause. We break total delay time into categories such as carrier, weather, security, and other recorded causes.

This helps move the analysis from **where delays happen** to **why they happen**, which is more actionable for operational decision-making.


In [30]:
con.sql("""
    WITH delay_causes AS (

        SELECT
            flight_year,
            'Carrier' AS delay_cause,
            SUM(COALESCE(DELAY_DUE_CARRIER, 0)) AS delay_minutes
        FROM flights_2019_2022
        WHERE completed_flight_flag = 1
          AND ARR_DELAY >= 15
        GROUP BY flight_year

        UNION ALL

        SELECT
            flight_year,
            'Weather',
            SUM(COALESCE(DELAY_DUE_WEATHER, 0))
        FROM flights_2019_2022
        WHERE completed_flight_flag = 1
          AND ARR_DELAY >= 15
        GROUP BY flight_year

        UNION ALL

        SELECT
            flight_year,
            'National Air System',
            SUM(COALESCE(DELAY_DUE_NAS, 0))
        FROM flights_2019_2022
        WHERE completed_flight_flag = 1
          AND ARR_DELAY >= 15
        GROUP BY flight_year

        UNION ALL

        SELECT
            flight_year,
            'Security',
            SUM(COALESCE(DELAY_DUE_SECURITY, 0))
        FROM flights_2019_2022
        WHERE completed_flight_flag = 1
          AND ARR_DELAY >= 15
        GROUP BY flight_year

        UNION ALL

        SELECT
            flight_year,
            'Late aircraft',
            SUM(COALESCE(DELAY_DUE_LATE_AIRCRAFT, 0))
        FROM flights_2019_2022
        WHERE completed_flight_flag = 1
          AND ARR_DELAY >= 15
        GROUP BY flight_year
    )

    SELECT
        flight_year,
        delay_cause,
        delay_minutes,
        ROUND(
            100.0 * delay_minutes
            / SUM(delay_minutes) OVER (PARTITION BY flight_year),
            2
        ) AS share_of_recorded_delay_minutes_pct

    FROM delay_causes
    ORDER BY flight_year, delay_minutes DESC
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,flight_year,delay_cause,delay_minutes,share_of_recorded_delay_minutes_pct
0,2019,Late aircraft,3855709.0,39.73
1,2019,Carrier,2937415.0,30.26
2,2019,National Air System,2355203.0,24.27
3,2019,Weather,543705.0,5.60
4,2019,Security,13825.0,0.14
5,2022,Carrier,3736174.0,39.76
6,2022,Late aircraft,3557967.0,37.87
7,2022,National Air System,1577937.0,16.79
8,2022,Weather,504925.0,5.37
9,2022,Security,19277.0,0.21


## 13. Question: When Is Flight Disruption Highest?

We analyze flight activity and delay performance by month to identify seasonal patterns. Understanding when disruption increases can help distinguish persistent operational issues from predictable periods of higher demand or adverse operating conditions.


In [28]:
con.sql("""
    SELECT
        flight_year,
        MONTH(FL_DATE) AS month_number,
        MONTHNAME(FL_DATE) AS month_name,

        COUNT(*) AS scheduled_flights,

        ROUND(
            100.0 * SUM(
                CASE
                    WHEN completed_flight_flag = 1
                     AND arrival_status = 'On time'
                    THEN 1
                    ELSE 0
                END
            ) / NULLIF(SUM(completed_flight_flag), 0),
            2
        ) AS on_time_arrival_rate_pct,

        ROUND(
            100.0 * SUM(CASE WHEN CANCELLED = 1 THEN 1 ELSE 0 END)
            / COUNT(*),
            2
        ) AS cancellation_rate_pct,

        ROUND(
            AVG(
                CASE
                    WHEN completed_flight_flag = 1
                    THEN ARR_DELAY
                END
            ),
            2
        ) AS average_arrival_delay_minutes

    FROM flights_2019_2022
    GROUP BY flight_year, MONTH(FL_DATE), MONTHNAME(FL_DATE)
    ORDER BY flight_year, month_number
""").df()

,flight_year,month_number,month_name,scheduled_flights,on_time_arrival_rate_pct,cancellation_rate_pct,average_arrival_delay_minutes
0,2019,1,January,59412,81.52,2.80,4.09
1,2019,2,February,54565,76.67,2.86,9.03
2,2019,3,March,64894,82.92,2.03,3.30
3,2019,4,April,62080,81.90,2.39,4.56
4,2019,5,May,64966,79.93,2.11,6.54
5,2019,6,June,64777,75.31,1.95,11.79
6,2019,7,July,67213,79.05,1.99,8.34
7,2019,8,August,67469,79.46,1.67,7.68
8,2019,9,September,61987,86.19,1.54,0.06
9,2019,10,October,64869,83.33,0.75,2.34


## 14. Prepare Analysis Outputs

The final step is to export the cleaned and derived datasets needed for downstream reporting and visualization.

These outputs are designed to make the analysis reusable in tools such as Power BI, allowing the operational findings developed here to be turned into an interactive business dashboard without rebuilding the underlying transformations.


In [29]:
from pathlib import Path

export_folder = Path(
    r"C:\Users\Sahil\OneDrive\Desktop\jupyter lab\Project 5\powerbi_data"
)
export_folder.mkdir(exist_ok=True)

cleaned_file = (export_folder / "flights_cleaned_2019_2022.csv").as_posix()

con.execute(f"""
    COPY (
        SELECT *
        FROM flights_2019_2022
    )
    TO '{cleaned_file}'
    (HEADER, DELIMITER ',')
""")

print(f"Created: {cleaned_file}")

Created: C:/Users/Sahil/OneDrive/Desktop/jupyter lab/Project 5/powerbi_data/flights_cleaned_2019_2022.csv
